# ColabUMA Scan — UMAによる緩和スキャン

Metaの機械学習ポテンシャル **UMA** で、結合長・角度・二面角を一定の刻みで動かしながら残りの自由度を最適化する **緩和スキャン（relaxed scan）** を行うノートブックです。Gaussian の `opt=modredundant` の `S` 指定に相当します。コードを書く必要はありません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. 入力フォーム** で出力名、電荷、スピン多重度、スキャン条件、収束判定を指定する
4. **ランタイム → すべて実行** を押す
5. **2. ファイル選択と事前確認** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
6. 一番下のセルでエネルギープロファイルと各点の構造を確認し、結果のZIPをダウンロードする

原子番号が分からないときは、そのまま一度実行してください。「5. 構造の準備」に番号付きの3D表示が出ます。番号を確認してフォームを直し、もう一度実行します。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

2回目以降は手順3〜6を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## スキャン条件の指定

| 項目 | 説明 |
|---|---|
| `scan_atoms` | 原子番号（1始まり）をスペース区切りで。2個なら結合長、3個なら角度、4個なら二面角 |
| `scan_start` | 開始値（Å または 度）。空欄なら入力構造の値から始める |
| `scan_n_steps` | 刻みの回数。計算する点の数はこれに1を足した数になる（Gaussian の `S` と同じ） |
| `scan_step_size` | 1回の刻み幅（Å または 度）。負の値で逆向きに動かす |
| `opt_before_scan` | オンなら、スキャンの前に入力構造を拘束なしで最適化する。`scan_start` が空欄なら、最適化後の値から始める |
| `opt_after_scan` | オンなら、スキャンの後に拘束なしで最適化する。開始構造はスキャンの最後の点からさかのぼって最初に見つかる極小 |

たとえば Gaussian の `D 1 2 3 4 S 36 10.0` は、`scan_atoms = 1 2 3 4`、`scan_n_steps = 36`、`scan_step_size = 10` です。

「後の最適化」は、行き過ぎたスキャンの端から戻って生成物側の極小を取るためのものです。たとえば結合を縮めすぎてエネルギーが上がっていても、その上り坂をさかのぼった先の極小から最適化します。0.1 kcal/mol 未満の小さな凹凸と未収束の点は無視します。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_scan.csv` | 各点の目標値、実測値、エネルギー、ΔE、fmax、ステップ数、収束（未収束の点も載せます） |
| `<出力名>_scan.xyz` | 収束した各点の最適化構造（複数構造xyz） |
| `<出力名>_scan.log` | 収束した各点の最終構造だけを Gaussian のスキャン出力の書式で書いたもの。Gaussian で計算した結果ではありません |
| `<出力名>_preopt.xyz` / `.log` | 前の最適化の結果と経路（オンのとき） |
| `<出力名>_postopt.xyz` / `.log` | 後の最適化の結果と経路（オンのとき） |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、各点と各ステップのエネルギーと fmax |

## 注意事項

- 各点は前の点の最適化構造から始めます。途中で構造が別の谷に飛ぶと、プロファイルに段差が出ます。逆向きにもスキャンして、同じ曲線になるか確認してください。
- プロファイルの最高点は遷移状態ではありません。遷移状態の初期構造の候補として使い、別途最適化して確認してください。
- 電荷とスピン多重度は全点で固定です。結合を切るスキャンでは、解離の極限でエネルギーが正しくならないことがあります。
- 環の中の座標をスキャンすると、指定の末端の原子だけを動かすため、各点の初期構造が大きくひずむことがあります。刻み幅を小さくしてください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください。
- 周期系（結晶・表面）には対応していません。
- スキャンの最高点やスキャン後の最適化構造から遷移状態を探すときは [ColabUMA_Workflow](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_Workflow.ipynb) を使うと、同じセッションのままスキャン → TS と続けられます。

## ライセンス

- このノートのコード：MIT License
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルで xyz または gjf ファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### スキャン条件
#@markdown 原子番号（1始まり）をスペース区切りで指定します。2個なら結合長、3個なら角度、4個なら二面角です。
scan_atoms = "1 2 3 4"  #@param {type:"string"}
#@markdown 開始値（Å または 度）。空欄なら入力構造の値から始めます（前の最適化がオンなら最適化後の値）。
scan_start = ""  #@param {type:"string"}
#@markdown 刻みの回数と刻み幅（Å または 度）。計算する点の数は、刻みの回数に1を足した数です。
scan_n_steps = 36  #@param {type:"integer"}
scan_step_size = 10.0  #@param {type:"number"}
#@markdown スキャンの前に、入力構造を拘束なしで最適化します。
opt_before_scan = True  #@param {type:"boolean"}
#@markdown スキャンの後に、最後の点からさかのぼって最初の極小を拘束なしで最適化します。
opt_after_scan = True  #@param {type:"boolean"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown 収束判定 fmax（eV/Å）。normal は Gaussian の既定、tight は Gaussian の opt=tight と同じくらいです。
convergence = "normal（0.02）"  #@param ["loose（0.05）", "normal（0.02）", "tight（0.001）", "数値指定"]
#@markdown 「数値指定」を選んだときの fmax（eV/Å）。
fmax_custom = 0.02  #@param {type:"number"}
#@markdown 最大ステップ数は1点あたりの値です。
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択とトークンの確認をここで済ませます。このあとのインストールと計算は放置できます。

import os
import subprocess
import sys

COLABUMA_REF = "main"  # 共通コード（colabuma パッケージ）を取ってくる GitHub のブランチ


def _use_colabuma(ref):
    # ノートと同じフォルダに colabuma があればそれを、なければ GitHub から取ってきて使う
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabUMA" if os.path.isdir("/content") else os.path.expanduser("~/ColabUMA")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabUMA.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"ColabUMA の共通コードを GitHub から取得できませんでした（ブランチ {ref}）。\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import env, opt, output, scan, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
spec = scan.parse_spec(scan_atoms, scan_start, scan_n_steps, scan_step_size, opt_before_scan, opt_after_scan)
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
scan.check_atom_range(spec, structure.quick_atom_count(upload_name, upload_data),
                      "フォームを直して、もう一度すべて実行してください。ファイル選択は Cancel upload で前回のファイルを使えます。")
ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール・認証） { display-mode: "form" }
DEVICE = env.setup()

In [ ]:
#@title 4. モデル読み込み { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. 構造の準備 { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"{atoms0.get_chemical_formula()}（{len(atoms0)} 原子）の初期構造を用意しました。")
sc = scan.prepare(spec, atoms0)

In [ ]:
#@title 6. スキャン { display-mode: "form" }
#@markdown 「収束」のバーは、fmax が最初の値から収束判定までに桁でどこまで近づいたかを示します。途中で fmax が上がるとバーも戻ります。
scan_res = scan.run(sc, atoms0, calc, job)

In [ ]:
#@title 7. 結果表示と保存 { display-mode: "form" }
files, log = scan.report(scan_res, job, atoms0)
files["log.txt"] = output.log_text("ColabUMA Scan 計算ログ", source_desc, job, [log])
zip_path = output.save_zip(job.out_prefix, files)